In [74]:
import os
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import datetime as dt
from data_processor import DataReader, DataPrep
from scipy import stats
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
nyse = mcal.get_calendar('NYSE')
# Get holidays
holidays = nyse.holidays().holidays

In [5]:
daily_data_path = r'data/daily_data'
intraday_data_path = r'data/intraday_data'
reader = DataReader()
intraday_df = reader.read_intraday_data(intraday_data_path)
daily_df = reader.read_daily_data(daily_data_path)
intraday_df.dropna(subset= 'CumReturnResid', inplace=True)

In [6]:
data_prep = DataPrep(intraday_df, daily_df)
target_df = data_prep.get_target(clip_MAD=True, normalize= True)
X_df = data_prep.get_features()

## Implement the Vanilla XGBoost model

In [23]:
print(len(X_df) - len(target_df))


874


In [27]:
input_data = X_df.join(target_df[["y"]],how="right")

In [17]:
target_df["y"].to_numpy()

array([ 0.05709568, -0.00229951, -0.02460268, ..., -0.08757397,
        0.02531936,  0.04150558])

In [30]:
input_data

Time  CumReturnResid  CumReturnRaw  CumVolume  \
Date       Id                                                                
2014-05-01 BBG000BT1715  15:30:00        0.008443      0.004344     348093   
           BBG000BJP882  15:30:00        0.001938     -0.004871     445807   
           BBG000DLVDK3  15:30:00        0.011024      0.001352    1894604   
           BBG000G8N9C6  15:30:00        0.001884      0.004570     600373   
           BBG000C90DH9  15:30:00       -0.003330     -0.003447    2803653   
...                           ...             ...           ...        ...   
2011-10-31 BBG000DMBXR2  15:30:00       -0.004918     -0.045781   33250772   
           BBG000BHSKN9  15:30:00        0.013291     -0.009298    1942987   
           BBG000GZQ728  15:30:00       -0.011187     -0.028714   16336481   
           BBG000BCTLF6  15:30:00        0.002672     -0.057183  177766560   
           BBG000B9XRY4  15:30:00        0.007804     -0.001197   11042327   

                          MDV_63_sqrt  Stock_Split  Dividend  \
Date       Id                                                  
2014-05-01 BBG000BT1715   8440.687768        False      True   
           BBG000BJP882   8291.020444        False     False   
           BBG000DLVDK3   8276.385685        False      True   
           BBG000G8N9C6   9624.988831        False      True   
           BBG000C90DH9   8298.258613        False      True   
...                               ...          ...       ...   
2011-10-31 BBG000DMBXR2  40007.428060        False     False   
           BBG000BHSKN9  41729.438050        False     False   
           BBG000GZQ728  43389.256735        False     False   
           BBG000BCTLF6  44422.953526        False     False   
           BBG000B9XRY4  88786.864456        False     False   

                         PxAdjFactorRatio  SharesAdjFactorRatio  \
Date       Id                                                     
2014-05-01 BBG000BT1715          1.021931                   1.0   
           BBG000BJP882          1.000000                   1.0   
           BBG000DLVDK3          1.061887                   1.0   
           BBG000G8N9C6          1.068860                   1.0   
           BBG000C90DH9          1.075747                   1.0   
...                                   ...                   ...   
2011-10-31 BBG000DMBXR2          1.000000                   1.0   
           BBG000BHSKN9          1.000000                   1.0   
           BBG000GZQ728          1.000000                   1.0   
           BBG000BCTLF6          1.000000                   1.0   
           BBG000B9XRY4          1.000000                   1.0   

                         Rolling_Return_5d  Rolling_Return_10d  \
Date       Id                                                    
2014-05-01 BBG000BT1715                NaN                 NaN   
           BBG000BJP882                NaN                 NaN   
           BBG000DLVDK3                NaN                 NaN   
           BBG000G8N9C6                NaN                 NaN   
           BBG000C90DH9                NaN                 NaN   
...                                    ...                 ...   
2011-10-31 BBG000DMBXR2           0.013704            0.026342   
           BBG000BHSKN9           0.015091            0.032218   
           BBG000GZQ728          -0.003480           -0.001168   
           BBG000BCTLF6           0.025045            0.027011   
           BBG000B9XRY4          -0.046799           -0.001470   

                         Rolling_Return_20d         y  
Date       Id                                          
2014-05-01 BBG000BT1715                 NaN  0.057096  
           BBG000BJP882                 NaN -0.002300  
           BBG000DLVDK3                 NaN -0.024603  
           BBG000G8N9C6                 NaN -0.028494  
           BBG000C90DH9                 NaN  0.215654  
...                                     ...       ...  
20

In [38]:
features = ["CumReturnResid", "Stock_Split", "Rolling_Return_5d", "Rolling_Return_10d", "Rolling_Return_20d"]

In [39]:
X_train, X_test, y_train, y_test = train_test_split(input_data[features], input_data["y"].to_numpy(), test_size=0.2, random_state=42, shuffle=False)


In [40]:
X_train

CumReturnResid  Stock_Split  Rolling_Return_5d  \
Date       Id                                                             
2014-05-01 BBG000BT1715        0.008443        False                NaN   
           BBG000BJP882        0.001938        False                NaN   
           BBG000DLVDK3        0.011024        False                NaN   
           BBG000G8N9C6        0.001884        False                NaN   
           BBG000C90DH9       -0.003330        False                NaN   
...                                 ...          ...                ...   
2011-01-14 BBG000PSKYX7       -0.006613        False           0.005726   
           BBG000BPWXK1       -0.009646        False          -0.071897   
           BBG000BHVJJ3        0.009365        False           0.015689   
           BBG000BLBVN4       -0.010178        False           0.021598   
           BBG000BCQZS4        0.015831        False           0.020360   

                         Rolling_Return_10d  Rolling_Return_20d  
Date       Id                                                    
2014-05-01 BBG000BT1715                 NaN                 NaN  
           BBG000BJP882                 NaN                 NaN  
           BBG000DLVDK3                 NaN                 NaN  
           BBG000G8N9C6                 NaN                 NaN  
           BBG000C90DH9                 NaN                 NaN  
...                                     ...                 ...  
2011-01-14 BBG000PSKYX7           -0.024936           -0.059128  
           BBG000BPWXK1           -0.068315           -0.163977  
           BBG000BHVJJ3           -0.028193           -0.015013  
           BBG000BLBVN4           -0.006453            0.013922  
           BBG000BCQZS4            0.046492           -0.000693  

[500445 rows x 5 columns]

In [42]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [43]:
params = {
    'max_depth': 3,  # the maximum depth of each tree
    'eta': 0.1,      # the training step for each iteration
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse'  # Root Mean Squared Error
}


In [44]:
num_boost_round = 100  # Number of boosting rounds
bst = xgb.train(params, dtrain, num_boost_round, [(dtest, 'test')], early_stopping_rounds=10)


[0]	test-rmse:0.06327
[1]	test-rmse:0.06327
[2]	test-rmse:0.06327
[3]	test-rmse:0.06327
[4]	test-rmse:0.06327
[5]	test-rmse:0.06327
[6]	test-rmse:0.06327
[7]	test-rmse:0.06327
[8]	test-rmse:0.06327
[9]	test-rmse:0.06327
[10]	test-rmse:0.06327
[11]	test-rmse:0.06327
[12]	test-rmse:0.06327
[13]	test-rmse:0.06327
[14]	test-rmse:0.06327


/Users/juliusgruber/anaconda3/envs/machinelearning/lib/python3.11/site-packages/xgboost/core.py:726: FutureWarning: Pass `evals` as keyword args.
  warnings.warn(msg, FutureWarning)


[15]	test-rmse:0.06327
[16]	test-rmse:0.06327


In [77]:
y_pred = bst.predict(dtest)

In [46]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse}")


RMSE: 0.06327056950526676


In [78]:
y_pred

array([-6.4677250e-04, -3.8766742e-04,  5.6935649e-04, ...,
        5.6935649e-04,  2.9653943e-06, -2.7973592e-04], dtype=float32)

In [82]:
r2_score(y_test, y_pred,sample_weight= daily_df[["EST_VOL"]].to_numpy())

ValueError: Found input variables with inconsistent numbers of samples: [125112, 125112, 629000]

In [85]:
y_pred

array([-6.4677250e-04, -3.8766742e-04,  5.6935649e-04, ...,
        5.6935649e-04,  2.9653943e-06, -2.7973592e-04], dtype=float32)

In [49]:
y_test

array([ 0.07384839,  0.08529373, -0.00942502, ..., -0.08757397,
        0.02531936,  0.04150558])

In [50]:
y_pred

array([-6.4677250e-04, -3.8766742e-04,  5.6935649e-04, ...,
        5.6935649e-04,  2.9653943e-06, -2.7973592e-04], dtype=float32)

In [62]:
daily_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 629000 entries, (Timestamp('2013-04-23 00:00:00'), 'BBG000BLCL55') to (Timestamp('2011-09-26 00:00:00'), 'BBG000B9XRY4')
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   SYMBOL                 629000 non-null  object 
 1   MIC                    629000 non-null  object 
 2   FREE_FLOAT_PERCENTAGE  629000 non-null  float64
 3   EST_VOL                626431 non-null  float64
 4   MDV_63                 629000 non-null  float64
 5   Open                   628992 non-null  float64
 6   High                   628992 non-null  float64
 7   Low                    628991 non-null  float64
 8   Close                  629000 non-null  float64
 9   Volume                 629000 non-null  float64
 10  PxAdjFactor            629000 non-null  float64
 11  SharesAdjFactor        629000 non-null  float64
dtypes: float64(10), object(2)
memory usage: 76.2+ MB


In [81]:
daily_df[["EST_VOL"]].to_numpy()

array([[0.21514],
       [0.10627],
       [0.17121],
       ...,
       [0.10913],
       [0.36069],
       [0.12022]])

In [69]:
daily_df = daily_df.sort_values(by='Date')

In [71]:
daily_df[["EST_VOL"]]

EST_VOL
Date       Id                   
2010-01-04 BBG000BG14P4  0.21514
           BBG000BQQH30  0.10627
           BBG000B9XRY4  0.17121
           BBG000BCTLF6  0.13670
           BBG000DMBXR2  0.10967
...                          ...
2014-12-31 BBG000BQG2C4  0.16108
           BBG000BNPSQ9  0.19833
           BBG000C7LMS8  0.10913
           BBG000BG14P4  0.36069
           BBG000BQD1J2  0.12022

[629000 rows x 1 columns]

In [73]:
?r2_score

Object `r2_score` not found.


In [51]:
import numpy as np

def weighted_r2(y_true, y_pred, weights):
    weighted_mean = np.sum(weights * y_true) / np.sum(weights)
    ss_res = np.sum(weights * (y_true - y_pred) ** 2)
    ss_tot = np.sum(weights * (y_true - weighted_mean) ** 2)
    r2_weighted = 1 - ss_res / ss_tot
    return r2_weighted

# Example usage:
weights = np.array([...])  # Your weights for each observation

r2_weighted = weighted_r2(y_true, y_pred, weights)
print(f"Weighted R^2: {r2_weighted}")


TypeError: unsupported operand type(s) for *: 'ellipsis' and 'ellipsis'

In [57]:
daily_df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 629000 entries, (Timestamp('2013-04-23 00:00:00'), 'BBG000BLCL55') to (Timestamp('2011-09-26 00:00:00'), 'BBG000B9XRY4')
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   SYMBOL                 629000 non-null  object 
 1   MIC                    629000 non-null  object 
 2   FREE_FLOAT_PERCENTAGE  629000 non-null  float64
 3   EST_VOL                626431 non-null  float64
 4   MDV_63                 629000 non-null  float64
 5   Open                   628992 non-null  float64
 6   High                   628992 non-null  float64
 7   Low                    628991 non-null  float64
 8   Close                  629000 non-null  float64
 9   Volume                 629000 non-null  float64
 10  PxAdjFactor            629000 non-null  float64
 11  SharesAdjFactor        629000 non-null  float64
dtypes: float64(10), object(2)
memory usage: 76.2+ MB
